# 🎛️ VGAIN SCAN ANALYSIS


**🔍 Step-by-Step**

**Interactive debugging**



In [ ]:
%load_ext autoreload
%autoreload 2

from waffles.coldboxVD.march_26.analysis.utils import *

In [ ]:
# RUN INPUT PARAMETERS

membrane = 'M3'
channel = 12
vgain = 900
led_value = 1100
bias = 784

In [ ]:
# ANALYSIS INPUT PARAMETERS 

## -- from input file --
# with open(f"/afs/cern.ch/work/a/anbalbon/private/waffles/src/waffles/coldboxVD/march_26/analysis/input/vgain_parameters_{membrane}_ch{channel}_bias{bias:04d}_led{led_value}.json", "r") as f:
#     dict_params_all = json.load(f)
# dict_params_all = {int(k): v for k, v in dict_params_all.items()}
# dict_params = dict_params_all[vgain] 

##-- from dict
dict_params = {
    "baseline_timeticks_limit": 120,
    "deviation_from_baseline": 0.6,
    "heatmap_min": -170,
    "heatmap_max": 550,
    "adcs_threshold": 50,
    "n_std_baseline": 1,
    "max_peaks": 7,
    "prominence": 0.5,
    "initial_percentage": 0.1,
    "percentage_step": 0.02,
    "ch_span_fraction_around_peaks": 0.05
  }

In [ ]:
# INPUT FILE
coldbox_folder = "/eos/experiment/neutplatform/protodune/experiments/ColdBoxVD/2026March/20260317_daphne-13_led_vgain_bias_scan_remote"
input_file = f"{coldbox_folder}/afe1bias_{bias:04d}/vgain_{vgain:04d}/led_{led_value}_ch{channel}/channel_{channel}.dat"

## 🔍 **Step-by-Step Analysis**


In [ ]:
wfset_original = create_waveform_set_from_spybuffer(filename=input_file, WFs=40000, length=1024, config_channel=channel)
# plotting_overlap_wf(wfset_original, index_list=[1,2])
# plotting_overlap_wf(wfset_original, n_wf=5)

REMOVING BASELINE

In [ ]:
# BASELINE ANALYSIS + removing 

baseliner_input_parameters = IPDict({
            'baseline_limits': (0,dict_params['baseline_timeticks_limit']),
            'std_cut': 1.,
            'type': 'mean'
        })

checks_kwargs = IPDict({
    'points_no': wfset_original.points_per_wf
})

baseline_analysis_label = 'baseline'

_ = wfset_original.analyse(
    baseline_analysis_label,
    WindowBaseliner,
    baseliner_input_parameters,
    checks_kwargs=checks_kwargs,
    overwrite=True
)

In [ ]:
wfset_original.apply(subtract_baseline, baseline_analysis_label, show_progress=False)
# plotting_overlap_wf(wfset_original, n_wf=2)

In [ ]:
# Dummy analysis for later - it sets the baseline to 0 always

null_baseline_analysis_label = 'null_baseliner'
_ = wfset_original.analyse(
            null_baseline_analysis_label,
            StoreWfAna,
            {'baseline': 0.},
            overwrite=True
        )

STARTING THE FILTERING PROCEDURE

By using the persistance plot, try to find a good sub wfset for the finger plot

In [ ]:
# No filter 

persistance_plot_helper(wfset_original, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)

In [ ]:
# First step - remove waveforms which goes below and up some adcs thresholds in the baseline region (similar to coarse_selection_for_led_calibration from Julio's code)

wfset_1 = WaveformSet.from_filtered_WaveformSet(wfset_original, adcs_threshold_filter, time_range = [0,dict_params['baseline_timeticks_limit']], adcs_minimum_threshold=-dict_params['adcs_threshold'], adcs_maximum_threshold=dict_params['adcs_threshold'])
persistance_plot_helper(wfset_1, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)

In [ ]:
# Second step - look at baseline std distribution to remove noisy waveforms -

average_baseline_std = compute_average_baseline_std(wfset_1, baseline_analysis_label)
#wfset_2 = WaveformSet.from_filtered_WaveformSet(wfset_1, baseline_std_selection, baseline_analysis_label, average_baseline_std, dict_params['n_std_baseline'])
wfset_2 = WaveformSet.from_filtered_WaveformSet(wfset_1, baseline_std_selection, baseline_analysis_label, average_baseline_std, 1)
persistance_plot_helper(wfset_2, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)

In [ ]:
print(f"Original wfset: {len(wfset_original.waveforms)}")
print(f"Adcs cut 1: {len(wfset_1.waveforms)}")
print(f"Std cut: {len(wfset_2.waveforms)}")

wfset_filtered = wfset_2

INTEGRATION WINDOW DEFINITION

Computing the mean waveform 

In [ ]:
# Selecting a subsample of waveforms without afterpulses for the mean waveform computation
wfset_filtered_meanwf = WaveformSet.from_filtered_WaveformSet(wfset_filtered, adcs_threshold_filter, time_range = [400,-1], adcs_minimum_threshold=-dict_params['adcs_threshold'], adcs_maximum_threshold=dict_params['adcs_threshold'])
persistance_plot_helper(wfset_filtered_meanwf, channel, ymin = dict_params['heatmap_min'], ymax = dict_params['heatmap_max'], adc_bins = 1000)


In [ ]:
mean_wf = wfset_filtered_meanwf.compute_mean_waveform()

plt.figure()
plt.plot(np.array(range(0,1024)), mean_wf.adcs, label="Mean wf")
plt.xlabel("Time ticks")
plt.ylabel("Adcs")
plt.title(f"Mean waveform")
plt.show()

In [ ]:
from typing import Tuple

from waffles.Exceptions import GenerateExceptionMessage

def my_get_pulse_window_limits(
    adcs_array: np.ndarray,
    baseline: float,
    deviation_from_baseline: float,
    lower_limit_correction: int = 0,
    upper_limit_correction: int = 0,
    get_zero_crossing_upper_limit: bool = False
) -> Tuple[int, int]:
    """This function takes an unidimensional numpy array
    representing the pulse signal, in ADCs, coming from an SiPM.
    It returns a tuple of two integers representing the lower
    and upper limits, respectively, of the SiPM pulse in the
    given array. The lower limit is unconditionally computed
    as the first point which negatively deviates from the given
    baseline by a certain given amount. The calculation of the
    upper limit depends on the get_zero_crossing_upper_limit
    parameter. If it set to False, then the upper limit is
    computed as the last point, after the minimum of the pulse,
    which deviates from the given baseline by more than the
    given amount. If it is set to True, then the upper limit
    is computed as the last point, after the minimum of the
    pulse, which stays below the baseline. Note that it is
    assumed that the pulse is negative.

    Parameters
    ----------
    adcs_array: numpy.ndarray
        The input signal to analyze
    baseline: float
        The baseline value of the signal. It must be greater
        than the minimum of the signal, i.e. np.min(adcs_array).
    deviation_from_baseline: float
        It must be greater than 0.0 and smaller than 1.0. It is
        interpreted as a fraction of baseline - np.min(adcs_array).
        I.e. the lower limit of the pulse window is assumed to
        be the first point which drops below x, where 

            x = baseline - \
                (deviation_from_baseline * (baseline - np.min(adcs_array))).

        Additionally, if get_zero_crossing_upper_limit is False,
        then the upper limit is assumed to be the last point, after
        the minimum of the pulse, which stays below x.
    lower_limit_correction: int
        This is an optional parameter that can be used to apply
        a correction, to the lower limit of the pulse window. I.e.
        if this parameter is set to N, then the final iterator
        value for the lower limit is the one inferred by the
        deviation-from-baseline analysis plus N. Note that N may
        be positive or negative.
    upper_limit_correction: int
        This parameter only makes a difference if
        get_zero_crossing_upper_limit is set to False. In that
        case, it is interpreted in the same way as
        lower_limit_correction, but it is applied to the upper
        limit of the pulse window.
    get_zero_crossing_upper_limit: bool
        If set to False, the upper limit of the pulse window
        is computed as the last point, after the minimum of
        the pulse, which deviates from the given baseline by
        more than the given amount. If it is set to True, then
        the upper limit is computed as the last point, after
        the minimum of the pulse, which stays below the
        baseline.

    Returns
    -------
    Tuple[int, int]
        A tuple containing the lower and upper limits of the
        pulse window.
    """

    idx_min = np.argmin(adcs_array)

    if baseline <= adcs_array[idx_min]:
        raise Exception(
            GenerateExceptionMessage(
                1,
                'get_pulse_window_limits()',
                f"The baseline ({baseline}) must be greater "
                f"than the minimum of the signal ({adcs_array[idx_min]})."
            )
        )
    
    if deviation_from_baseline <= 0.0 or deviation_from_baseline >= 1.0:
        raise Exception(
            GenerateExceptionMessage(
                2,
                'get_pulse_window_limits()',
                "The deviation_from_baseline parameter "
                f"({deviation_from_baseline}) must belong to the (0.0, 1.0)"
                " range."
            )
        )
   
    idx_max = np.argmax(adcs_array)
    threshold = (deviation_from_baseline * np.abs(adcs_array[idx_max]))
    
    print(f"threshold: {threshold}")
    print(f"baseline {baseline}")
    print(f"min {adcs_array[idx_min]}")
    print(f"dev {deviation_from_baseline}")

    # Compute the lower limit of the pulse window
    lower_limit = None
    for i, adc in enumerate(adcs_array):
        if adc > threshold:
            lower_limit = i
            break

    # Since we've made sure that baseline > adcs_array[idx_min],
    # lower_limit must be defined, i.e. an adcs value smaller
    # than the threshold must have been found (even if it matches
    # the minimum of the signal). The fact that idx_min is bigger
    # or equal to lower_limit is also guaranteed.

    upper_limit = None
    if not get_zero_crossing_upper_limit:
        for i in range(idx_min + 1, len(adcs_array)):
            if adcs_array[i] < threshold:
                upper_limit = i - 1
                break
    else:
        for i in range(idx_min + 1, len(adcs_array)):
            if adcs_array[i] < baseline:
                upper_limit = i - 1
                break

    if upper_limit is None:
        warning_message = "The upper limit of the pulse window "\
        "could not be found. I.e. the signal never raised above the "

        if not get_zero_crossing_upper_limit:
            warning_message += f"threshold ({threshold}) "
        else:
            warning_message += f"baseline ({baseline}) "

        warning_message += f"after reaching its minimum value "\
        f"({adcs_array[idx_min]}). The upper limit will be set "\
        f"to the last point of the array ({len(adcs_array) - 1})."

        print(
            "In function get_pulse_window_limits(): "
            f"WARNING: {warning_message}"
        )

        upper_limit = len(adcs_array) - 1

    # Apply the corrections to the limits
    lower_limit += lower_limit_correction
    if not get_zero_crossing_upper_limit:
        upper_limit += upper_limit_correction

    # Make sure that the corrected limits are within the bounds of the array
    lower_limit, upper_limit = max(0, lower_limit), max(0, upper_limit)
    lower_limit, upper_limit = min(len(adcs_array) - 1, lower_limit), \
        min(len(adcs_array) - 1, upper_limit)

    # Make sure that the corrections did not make the lower limit
    # greater than or equal to the upper limit
    if lower_limit == upper_limit:
        print(
            "In function get_pulse_window_limits(): "
            "WARNING: The corrected lower limit of the pulse window "
            f"({lower_limit}) is equal to the corrected upper limit "
            f"({upper_limit}). The upper limit will be set to "
            f"{upper_limit + 1}."
        )
        upper_limit += 1

    elif lower_limit > upper_limit:
        raise Exception(
            GenerateExceptionMessage(
                3,
                'get_pulse_window_limits()',
                "The corrected lower limit of the pulse window "
                f"({lower_limit}) is greater than the corrected "
                f"upper limit ({upper_limit})."
            )
        )

    return (
        lower_limit,
        upper_limit
    )

In [ ]:
dict_params['deviation_from_baseline'] = 0.3

aux_limits = my_get_pulse_window_limits(
                    adcs_array = mean_wf.adcs,
                    baseline = 0,
                    deviation_from_baseline = dict_params['deviation_from_baseline'],
                    get_zero_crossing_upper_limit = False
                )

print(aux_limits)

In [ ]:
plt.figure()
plt.plot(np.array(range(0,1024)), mean_wf.adcs, label="Mean wf")
plt.axvline(
    aux_limits[0],
    linestyle="--",
    color="red",
    linewidth=1,
    label=f"My LL = {aux_limits[0]} ({dict_params['deviation_from_baseline']:.1f} σ)"
)

plt.axvline(
    aux_limits[1],
    linestyle="--",
    color="blue",
    linewidth=1,
    label=f"My UL = {aux_limits[1]} ({dict_params['deviation_from_baseline']:.1f} σ)"
)

plt.axhline(dict_params['deviation_from_baseline'] * np.abs(0 - adcs_array[idx_min]))

# federico_limits = (382, 406)
#plt.axvline(
   #federico_limits[0],
   # linestyle="-",
   # color="red",
   # linewidth=1,
   # label=f"Federico LL = {federico_limits[0]}"
#)

#plt.axvline(
    #federico_limits[1],
    #linestyle="-",
    #color="blue",
    #linewidth=1,
    #label=f"Federico UL = {federico_limits[1]}"
#

plt.legend()
plt.xlabel("Time ticks (AU)")
plt.ylabel("Adcs")
plt.title(f"Mean waveform")
# plt.xlim(350,500)
plt.show()

In [ ]:
# Decide if you want to use your or Federico's limits for integration
# aux_limits = federico_limits

INTEGRATION ANALYSIS

In [ ]:
integration_analysis_label = 'integration_analysis'

integrator_input_parameters = IPDict({
        'baseline_analysis': null_baseline_analysis_label,
        'inversion': False,
        'int_ll': aux_limits[0],
        'int_ul': aux_limits[1],
        'amp_ll': aux_limits[0],
        'amp_ul': aux_limits[1]
    })

checks_kwargs = IPDict({'points_no': wfset_filtered.points_per_wf})

_ = wfset_filtered.analyse(
    integration_analysis_label,
    WindowIntegrator,
    integrator_input_parameters,
    checks_kwargs=checks_kwargs,
    overwrite=True
)

CALIBRATION HISTOGRAM 

In [ ]:
hist_domain, hist_nbins, hist_bins_width = auto_histogram(wfset_filtered, integration_analysis_label, show_results=True)

In [ ]:
my_grid = coldbox_single_channel_grid(wfset_filtered, config_channel=channel)

my_grid.compute_calib_histos(
            bins_number=hist_nbins, 
            domain=hist_domain, 
            variable='integral',
            analysis_label=integration_analysis_label
        )

fit_peaks_of_ChannelWsGrid(
        my_grid,
        max_peaks=dict_params['max_peaks'], 
        prominence=float(dict_params['prominence']), 
        initial_percentage=dict_params['initial_percentage'],
        percentage_step=dict_params['percentage_step'],
        return_last_addition_if_fail=True,
        fit_type='multigauss_iminuit',
        weigh_fit_by_poisson_sigmas=True,
        ch_span_fraction_around_peaks=dict_params['ch_span_fraction_around_peaks']
    )

# # If you want to play on parameters manually
# fit_peaks_of_ChannelWsGrid(
#     my_grid,
#     max_peaks=7,
#     prominence=0.5,
#     initial_percentage=0.1,
#     percentage_step=0.02,
#     return_last_addition_if_fail=True,
#     fit_type='multigauss_iminuit',
#     weigh_fit_by_poisson_sigmas=True,
#     ch_span_fraction_around_peaks=0.05
# )


fig = plot_ChannelWsGrid(
    my_grid, 
    mode='calibration',
    plot_peaks_fits=True,           
    plot_sum_of_gaussians=True      
)

fig.show()

output_parameters = print_correlated_gaussians_fit_parameters(my_grid, federico_conversion=True, show=True)
